# Appendix Tables — WaveletFlow Paper

## 0. Imports & Global Settings

In [1]:
import glob
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

rc = {
    "axes.axisbelow":      True,
    "axes.linewidth":      1.0,
    "axes.labelsize":      "x-small",
    "axes.titlesize":      "x-small",
    "figure.dpi":          150,
    "figure.figsize":      (3.2, 3.2),
    "font.size":           12.0,
    "legend.fontsize":     "xx-small",
    "lines.linewidth":     1.0,
    "savefig.transparent": True,
    "xtick.labelsize":     "xx-small",
    "xtick.major.width":   1.0,
    "ytick.labelsize":     "xx-small",
    "ytick.major.width":   1.0,
    "font.family":         ["serif"],
    "text.usetex":         True,
    "text.latex.preamble": r"\usepackage{amsfonts}",
}
plt.rcParams.update(rc)

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

## 1. Model Registry & Derived Sets

In [2]:
def rgb(r, g, b): return f"#{r:02x}{g:02x}{b:02x}"

# Each entry: label, color, dash, lw, group in {wfm-1scale, wfm-3scale, generative, deterministic}
MODEL_REGISTRY = {
    # -- WFM 1-scale variants --------------------------------------------------
    "WaveletFlow_1scale_haar_v4.0": dict(label="WFM-1 haar", color=rgb( 98,124,107), dash=(1,0), lw=2.5, group="wfm-1scale"),
    "WaveletFlow_1scale_db2_v4.0":  dict(label="WFM-1 db2",  color=rgb( 98,124,107), dash=(1,0), lw=2.5, group="wfm-1scale"),
    "WaveletFlow_1scale_db4_v4.0":  dict(label="WFM-1 db4",  color=rgb( 98,124,107), dash=(1,0), lw=2.5, group="wfm-1scale"),
    "WaveletFlow_1scale_db6_v4.0":  dict(label="WFM-1 db6",  color=rgb( 98,124,107), dash=(1,0), lw=2.5, group="wfm-1scale"),
    # -- WFM 3-scale variants --------------------------------------------------
    "WaveletFlow_3scales_haar_v4.0": dict(label="WFM-3 haar", color=rgb(117,176,138), dash=(1,0), lw=2.5, group="wfm-3scale"),
    "WaveletFlow_3scales_db2_v4.0":  dict(label="WFM-3 db2",  color=rgb(117,176,138), dash=(1,0), lw=2.5, group="wfm-3scale"),
    "WaveletFlow_3scales_db4_v4.0":  dict(label="WFM-3 db4",  color=rgb(117,176,138), dash=(1,0), lw=2.5, group="wfm-3scale"),
    "WaveletFlow_3scales_db6_v4.0":  dict(label="WFM-3 db6",  color=rgb(117,176,138), dash=(1,0), lw=2.5, group="wfm-3scale"),
    # -- Generative baselines --------------------------------------------------
    "FM_data_space_v4.0":       dict(label=r"FM$_\text{pixel}$",   color=rgb(176,176,176), dash=(1,0), lw=1.8, group="generative"),
    "FourierFlow_32modes_v4.0": dict(label=r"FM$_\text{fourier}$", color=rgb(143, 94,106), dash=(1,0), lw=1.8, group="generative"),
    "WDNO":                     dict(label="WDNO",                  color=rgb(191,134,148), dash=(1,0), lw=1.8, group="generative"),
    # -- Deterministic baselines -----------------------------------------------
    "WNO_3scales_db2": dict(label="WNO",  color=rgb(176,136, 84), dash=(1,0), lw=1.8, group="deterministic"),
    "FNO":             dict(label="FNO",  color=rgb(208,169,110), dash=(1,0), lw=1.8, group="deterministic"),
    "TFNO":            dict(label="TFNO", color=rgb(236,227,163), dash=(1,0), lw=1.8, group="deterministic"),
}

all_models = list(MODEL_REGISTRY)
WFM_1scale = [m for m in all_models if MODEL_REGISTRY[m]["group"] == "wfm-1scale"]
WFM_3scale = [m for m in all_models if MODEL_REGISTRY[m]["group"] == "wfm-3scale"]
WFM_all    = WFM_1scale + WFM_3scale
gen_models = [m for m in all_models if MODEL_REGISTRY[m]["group"] == "generative"]
det_models = [m for m in all_models if MODEL_REGISTRY[m]["group"] == "deterministic"]

# Canonical table ordering: WFM-1scale -> WFM-3scale -> generative -> deterministic
MODEL_ORDER = WFM_1scale + WFM_3scale + gen_models + det_models

labels     = {m: MODEL_REGISTRY[m]["label"] for m in all_models}
palette    = {m: MODEL_REGISTRY[m]["color"] for m in all_models}
dashes     = {m: MODEL_REGISTRY[m]["dash"]  for m in all_models}
linewidths = {m: MODEL_REGISTRY[m]["lw"]    for m in all_models}

GT_LABEL = "Ground-truth"
palette[GT_LABEL] = "#222222"; dashes[GT_LABEL] = (1, 0)
labels[GT_LABEL]  = GT_LABEL;  linewidths[GT_LABEL] = 1.8

def model_group(m):
    if m in WFM_1scale: return "wfm-1scale"
    if m in WFM_3scale: return "wfm-3scale"
    if m in gen_models: return "generative"
    return "deterministic"

## 2. PDE Metadata

In [3]:
PDEs = ["turbulent_radiative_layer_2D", "rayleigh_benard", "active_matter"]

PDE_LABELS = {
    "turbulent_radiative_layer_2D": "Turbulence",
    "rayleigh_benard":              r"Rayleigh--B\'{e}nard",
    "active_matter":                "Active Matter",
}

# Physical parameters varied in ablation studies
PDE_CONSTANTS = {
    "turbulent_radiative_layer_2D": ["tcool"],
    "rayleigh_benard":              ["Rayleigh", "Prandtl"],
    "active_matter":                ["zeta", "alpha"],
}

# Raw horizon strings produced by the step->horizon mapping (per PDE)
HORIZON_BINS_RAW = {
    "turbulent_radiative_layer_2D": ["01:32", "33:64", "65:97"],
    "rayleigh_benard":              ["01:32", "33:64", "65:196"],
    "active_matter":                ["01:16", "17:32", "33:77"],
}

# Generic column labels for unified multi-PDE tables (Short / Mid / Long)
GENERIC_HORIZONS = ["Short", "Mid", "Long"]
HORIZON_TO_GENERIC = {
    pde: dict(zip(raw, GENERIC_HORIZONS))
    for pde, raw in HORIZON_BINS_RAW.items()
}

# Spectral frequency bands
BANDS     = ["Low", "Mid", "High"]
BAND_COLS = {"rmse_c_low": "Low", "rmse_c_mid": "Mid", "rmse_c_high": "High"}

# Horizon caption note (reused in LaTeX exports)
HZ_CAPTION_NOTE = (
    r"Short/Mid/Long $\equiv$ [1\!:\!32]/[33\!:\!64]/[65\!:\!97] (Turbulence), "
    r"[1\!:\!32]/[33\!:\!64]/[65\!:\!196] (Rayleigh--B\'enard), "
    r"[1\!:\!16]/[17\!:\!32]/[33\!:\!77] (Active Matter)."
)

## 3. Data Loading

In [4]:
LOAD_STATS = True
LOAD_PSD   = False


def enrich(df, stem, model_name, dataset_name):
    # Preserve GT rows already labelled in the CSV
    parts     = stem.split("_")
    nsampling = next((p.replace("nsampling", "") for p in parts if p.startswith("nsampling")), None)
    nens      = next((p.replace("nens",      "") for p in parts if p.startswith("nens")),      None)
    if "model" in df.columns:
        df.loc[df["model"] != GT_LABEL, "model"] = model_name
    else:
        df["model"] = model_name
    df["source"]    = stem
    df["dataset"]   = dataset_name
    df["pde"]       = dataset_name
    df["nsampling"] = int(nsampling) if nsampling else None
    df["nens"]      = int(nens)      if nens      else None
    return df


DEDUP_EXCLUDE = {"source", "nsampling", "nens"}

dfs_stats, dfs_psd = [], []
for pde in PDEs:
    print(f"\n{'='*60}\nLoading: {pde}\n{'='*60}")
    for model in all_models:
        for suffix, target, flag in [
            ("stats", dfs_stats, LOAD_STATS),
            ("psd",   dfs_psd,   LOAD_PSD),
        ]:
            if not flag:
                continue
            pattern = f"{PROJECT_ROOT}/experiments/{pde}/metrics/{model}*{suffix}.csv"
            matched = sorted(glob.glob(pattern))
            if not matched:
                print(f"  [MISS {suffix}] {model}")
            for fpath in matched:
                try:
                    df = pd.read_csv(fpath)
                    target.append(enrich(df, Path(fpath).stem, model, pde))
                    print(f"  [OK   {suffix}] {Path(fpath).name}")
                except Exception as exc:
                    print(f"  [ERR] {fpath}: {exc}")


def _concat_dedup(dfs):
    if not dfs:
        return pd.DataFrame()
    full = pd.concat(dfs, ignore_index=True)
    return full.drop_duplicates(
        subset=[c for c in full.columns if c not in DEDUP_EXCLUDE]
    )


stats_df = _concat_dedup(dfs_stats)
psd_df   = _concat_dedup(dfs_psd)
print(f"\nstats_df : {stats_df.shape}   cols: {list(stats_df.columns)}")
print(f"psd_df   : {psd_df.shape}   cols: {list(psd_df.columns)}")


Loading: turbulent_radiative_layer_2D
  [OK   stats] WaveletFlow_1scale_haar_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_1scale_db2_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_1scale_db4_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_1scale_db6_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_3scales_haar_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_3scales_db2_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_3scales_db4_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_3scales_db6_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] FM_data_space_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] FourierFlow_32modes_v4.0_nsampling50_nens8_turbul

## 4. Feature Engineering

In [5]:
AGG_MODE = "mean"   # "mean" | "median"

def aggregate(x):
    return np.nanmean(x) if AGG_MODE == "mean" else np.nanmedian(x)


def _horizon_fn(bins):
    # Build step -> horizon_label function from ordered (threshold, label) pairs
    def fn(t):
        for thresh, label in bins:
            if t <= thresh:
                return label
        return bins[-1][1]
    return fn


HORIZON_FN = {
    "turbulent_radiative_layer_2D": _horizon_fn([(32,"01:32"), (64,"33:64"), (97, "65:97")]),
    "rayleigh_benard":              _horizon_fn([(32,"01:32"), (64,"33:64"), (196,"65:196")]),
    "active_matter":                _horizon_fn([(16,"01:16"), (32,"17:32"), (77, "33:77")]),
}

if LOAD_STATS and not stats_df.empty:
    stats_df = stats_df.copy()
    stats_df["step"] += 1
    stats_df["horizon"] = pd.NA
    for pde, fn in HORIZON_FN.items():
        mask = stats_df["pde"] == pde
        stats_df.loc[mask, "horizon"] = stats_df.loc[mask, "step"].apply(fn)
    stats_df["model_group"] = stats_df["model"].apply(model_group)
    if {"spread", "rmse"}.issubset(stats_df.columns):
        stats_df["spread_skill"] = (stats_df["spread"] + 1e-3) / (stats_df["rmse"] + 1e-3)

if LOAD_PSD and not psd_df.empty:
    psd_df = psd_df.copy()
    psd_df["step"] += 1
    psd_df["horizon"] = pd.NA
    for pde, fn in HORIZON_FN.items():
        mask = psd_df["pde"] == pde
        psd_df.loc[mask, "horizon"] = psd_df.loc[mask, "step"].apply(fn)
    psd_df["model_group"] = psd_df["model"].apply(model_group)

## 5. Styling & LaTeX Helpers

In [6]:
# ============================================================
# Jupyter display helpers
# ============================================================

def highlight_min(df: pd.DataFrame) -> pd.DataFrame:
    # Bold minimum, underline 2nd minimum, per column
    out = pd.DataFrame("", index=df.index, columns=df.columns)
    for col in df.columns:
        vals = df[col].dropna().sort_values()
        if len(vals) >= 1: out.loc[vals.index[0], col] = "font-weight: bold"
        if len(vals) >= 2: out.loc[vals.index[1], col] = "text-decoration: underline"
    return out


def highlight_min_pde(df: pd.DataFrame) -> pd.DataFrame:
    # Bold minimum, underline 2nd minimum, per column, WITHIN each PDE group
    out = pd.DataFrame("", index=df.index, columns=df.columns)
    for pde in df.index.get_level_values(0).unique():
        mask = df.index.get_level_values(0) == pde
        sub  = df.loc[mask]
        for col in df.columns:
            vals = sub[col].dropna().sort_values()
            if len(vals) >= 1: out.loc[vals.index[0], col] = "font-weight: bold"
            if len(vals) >= 2: out.loc[vals.index[1], col] = "text-decoration: underline"
    return out


# ============================================================
# LaTeX helpers
# ============================================================

def _fmt_cell(val: float, rank: int) -> str:
    s = f"{val:.3f}"
    if rank == 1: return r"\textbf{" + s + "}"
    if rank == 2: return r"\underline{" + s + "}"
    return s


def to_latex_multiindex(
    df: pd.DataFrame,
    caption: str,
    label: str,
    minimize: bool = True,
    header_lines: list = None,
) -> str:
    # Export a (PDE, Model) 2-level MultiIndex DataFrame to a LaTeX booktabs table.
    #
    # Layout:
    #   - PDE groups are delimited by \midrule.
    #   - The PDE cell spans its group rows via \multirow + \rotatebox{90}.
    #   - Bold  = best  per column *within* the PDE block.
    #   - Underline = 2nd best per column *within* the PDE block.
    assert isinstance(df.index, pd.MultiIndex) and df.index.nlevels == 2

    # Per-PDE-block column ranks:  {(pde_label, model_label, col): rank}
    ranks = {}
    for pde in df.index.get_level_values(0).unique():
        mask = df.index.get_level_values(0) == pde
        sub  = df.loc[mask]
        for col in df.columns:
            sorted_idx = sub[col].dropna().sort_values(ascending=minimize).index.tolist()
            for rank, idx in enumerate(sorted_idx, 1):
                ranks[(pde, idx[1], col)] = rank   # idx = (pde_label, model_label)

    pde_counts = {
        pde: int((df.index.get_level_values(0) == pde).sum())
        for pde in df.index.get_level_values(0).unique()
    }

    n_cols   = len(df.columns)
    col_spec = "ll" + "c" * n_cols

    if header_lines is None:
        header_lines = [
            "PDE & Model & " + " & ".join(str(c) for c in df.columns) + r" \\"
        ]

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        r"\small",
        r"\begin{tabular}{" + col_spec + "}",
        r"\toprule",
        *header_lines,
        r"\midrule",
    ]

    prev_pde = None
    for (pde, model), row in df.iterrows():
        if prev_pde is not None and pde != prev_pde:
            lines.append(r"\midrule")
        pde_cell = (
            r"\multirow{" + str(pde_counts[pde]) + r"}{*}{\rotatebox{90}{" + pde + "}}"
            if pde != prev_pde else ""
        )
        prev_pde = pde
        cells = [pde_cell, model]
        for col in df.columns:
            val  = row[col]
            rank = ranks.get((pde, model, col), 99)
            cells.append("--" if pd.isna(val) else _fmt_cell(val, rank))
        lines.append(" & ".join(cells) + r" \\")

    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\caption{" + caption + "}",
        r"\label{" + label + "}",
        r"\end{table}",
    ]
    return "\n".join(lines)


def to_latex_simple(
    df: pd.DataFrame,
    caption: str,
    label: str,
    minimize: bool = True,
    header_lines: list = None,
) -> str:
    # Export a flat-index DataFrame to a LaTeX booktabs table.
    # Bold = best, underline = 2nd best, per column.
    ranks = {}
    for col in df.columns:
        sorted_idx = df[col].dropna().sort_values(ascending=minimize).index.tolist()
        ranks[col] = {m: i + 1 for i, m in enumerate(sorted_idx)}

    n_cols = len(df.columns)

    if header_lines is None:
        header_lines = [
            "Model & " + " & ".join(str(c) for c in df.columns) + r" \\"
        ]

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        r"\small",
        r"\begin{tabular}{l" + "c" * n_cols + "}",
        r"\toprule",
        *header_lines,
        r"\midrule",
    ]
    for model, row in df.iterrows():
        cells = [str(model)]
        for col in df.columns:
            val  = row[col]
            rank = ranks[col].get(model, 99)
            cells.append("--" if pd.isna(val) else _fmt_cell(val, rank))
        lines.append(" & ".join(cells) + r" \\")
    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\caption{" + caption + "}",
        r"\label{" + label + "}",
        r"\end{table}",
    ]
    return "\n".join(lines)

## 6. Table Builders

In [7]:
# ============================================================
# Main appendix tables
# ============================================================

def build_main_table(
    metric: str,
    models: list = None,
    pdes: list = None,
) -> pd.DataFrame:
    # Build a unified multi-PDE appendix table.
    #
    # Rows : 2-level MultiIndex  (PDE display label,  model display label)
    #        ordered by MODEL_ORDER within each PDE block.
    # Cols : Short / Mid / Long  (generic horizon labels, consistent across PDEs).
    #
    # metric : stats_df column to aggregate (e.g. "vrmse", "crps").
    # models : ordered subset of model keys; defaults to MODEL_ORDER.
    # pdes   : ordered subset of PDE keys;   defaults to PDEs.
    if models is None: models = MODEL_ORDER
    if pdes   is None: pdes   = PDEs

    frames = []
    for pde in pdes:
        sub = (
            stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
            [["model", "horizon", metric]]
            .groupby(["model", "horizon"])[metric]
            .agg(aggregate).round(3).reset_index()
        )
        sub["horizon"] = sub["horizon"].map(HORIZON_TO_GENERIC[pde])
        sub = sub.pivot(index="model", columns="horizon", values=metric)
        sub = sub.reindex(
            index=[m for m in models if m in sub.index],
            columns=[h for h in GENERIC_HORIZONS if h in sub.columns],
        )
        sub.index = pd.MultiIndex.from_tuples(
            [(PDE_LABELS[pde], labels.get(m, m)) for m in sub.index],
            names=["PDE", "Model"],
        )
        frames.append(sub)
    return pd.concat(frames)


def build_spectral_table(
    models: list = None,
    pdes: list = None,
) -> pd.DataFrame:
    # Multi-PDE spectral coherence RMSE table (averaged over horizons).
    # Rows : (PDE, model) MultiIndex
    # Cols : Low / Mid / High
    if models is None: models = MODEL_ORDER
    if pdes   is None: pdes   = PDEs

    frames = []
    for pde in pdes:
        sub = (
            stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
            [["model", "horizon"] + list(BAND_COLS.keys())]
            .groupby(["model", "horizon"])[list(BAND_COLS.keys())]
            .agg(aggregate).reset_index()
            .rename(columns=BAND_COLS)
            .groupby("model")[BANDS]
            .mean().round(3)
        )
        sub = sub.reindex([m for m in models if m in sub.index])
        sub.index = pd.MultiIndex.from_tuples(
            [(PDE_LABELS[pde], labels.get(m, m)) for m in sub.index],
            names=["PDE", "Model"],
        )
        frames.append(sub)
    return pd.concat(frames)


def build_spectral_horizon_table(
    models: list = None,
    pdes: list = None,
) -> pd.DataFrame:
    # Per-PDE spectral table broken down by horizon AND band.
    # Rows : (PDE, model) MultiIndex
    # Cols : (band, horizon) MultiIndex
    if models is None: models = MODEL_ORDER
    if pdes   is None: pdes   = PDEs

    frames = []
    for pde in pdes:
        sub = (
            stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
            [["model", "horizon"] + list(BAND_COLS.keys())]
            .melt(id_vars=["model", "horizon"],
                  value_vars=list(BAND_COLS.keys()),
                  var_name="band_key", value_name="srmse")
            .assign(band=lambda x: x["band_key"].map(BAND_COLS),
                    horizon=lambda x: x["horizon"].map(HORIZON_TO_GENERIC[pde]))
            .groupby(["model", "band", "horizon"])["srmse"]
            .agg(aggregate).round(3).reset_index()
            .pivot(index="model", columns=["band", "horizon"], values="srmse")
            .reindex(columns=BANDS, level="band")
            .reindex(columns=GENERIC_HORIZONS, level="horizon")
        )
        sub = sub.reindex([m for m in models if m in sub.index])
        sub.index = pd.MultiIndex.from_tuples(
            [(PDE_LABELS[pde], labels.get(m, m)) for m in sub.index],
            names=["PDE", "Model"],
        )
        frames.append(sub)
    return pd.concat(frames)


def build_w1_table(
    models: list = None,
    pdes: list = None,
) -> pd.DataFrame:
    # Wasserstein-1 table: (PDE, model) rows x single W1 column.
    if models is None: models = WFM_all + gen_models
    if pdes   is None: pdes   = PDEs

    frames = []
    for pde in pdes:
        sub = (
            stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
            [["model", "w1"]]
            .groupby("model")["w1"]
            .agg(aggregate).round(3)
            .reset_index().set_index("model")
        )
        sub = sub.reindex([m for m in models if m in sub.index])
        sub.index = pd.MultiIndex.from_tuples(
            [(PDE_LABELS[pde], labels.get(m, m)) for m in sub.index],
            names=["PDE", "Model"],
        )
        sub.columns = [r"$W_1$"]
        frames.append(sub)
    return pd.concat(frames)


# ============================================================
# Ablation tables
# ============================================================

def build_ablation_table(
    metric: str,
    pde: str,
    param: str,
    models: list = None,
    param_filter=None,
) -> pd.DataFrame:
    # Ablation table: model rows x param-value columns.
    #
    # metric       : "vrmse", "crps", etc.
    # pde          : PDE key string.
    # param        : physical parameter column (e.g. "tcool", "Rayleigh", "alpha").
    # models       : models to show; defaults to WFM_all + gen_models.
    # param_filter : optional callable applied to param values,
    #                e.g. ``lambda v: v <= 0.03`` to restrict tcool range.
    if models is None: models = WFM_all + gen_models

    sub = (
        stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
        [["model", param, metric]]
        .dropna(subset=[param, metric])
    )
    if param_filter is not None:
        sub = sub[sub[param].apply(param_filter)]

    sub = (
        sub.groupby(["model", param])[metric]
        .agg(aggregate).round(3).reset_index()
        .pivot(index="model", columns=param, values=metric)
    )
    sub = sub.reindex([m for m in models if m in sub.index])
    sub.index = [labels.get(m, m) for m in sub.index]
    sub.index.name  = "Model"
    sub.columns.name = param
    return sub


def build_spectral_ablation_table(
    pde: str,
    param: str,
    models: list = None,
    param_filter=None,
) -> pd.DataFrame:
    # Spectral ablation table: model rows x (band, param_value) columns.
    if models is None: models = WFM_all + gen_models

    sub = (
        stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
        [["model", param] + list(BAND_COLS.keys())]
        .dropna(subset=[param])
    )
    if param_filter is not None:
        sub = sub[sub[param].apply(param_filter)]

    sub = (
        sub.melt(id_vars=["model", param],
                 value_vars=list(BAND_COLS.keys()),
                 var_name="band_key", value_name="srmse")
        .assign(band=lambda x: x["band_key"].map(BAND_COLS))
        .groupby(["model", "band", param])["srmse"]
        .agg(aggregate).round(3).reset_index()
        .pivot(index="model", columns=["band", param], values="srmse")
        .reindex(columns=BANDS, level="band")
    )
    sub = sub.reindex([m for m in models if m in sub.index])
    sub.index = [labels.get(m, m) for m in sub.index]
    sub.index.name = "Model"
    return sub

---
# A. Main Appendix Tables

**Row structure**: 2-level MultiIndex `(PDE, Model)`.
WFM-1scale variants first (haar → db2 → db4 → db6), then WFM-3scale, then generative baselines, then deterministic baselines.

**Column structure**: Short / Mid / Long horizons (main tables) or Low / Mid / High frequency bands (spectral).

**Highlighting**: **bold** = best, <u>underline</u> = 2nd best, ranked *within each PDE block*.

## A.1  VRMSE — All Models

In [8]:
t_vrmse = build_main_table(metric="vrmse", models=MODEL_ORDER)
t_vrmse.style.apply(highlight_min_pde, axis=None).format("{:.3f}", na_rep="--")

/var/folders/ch/y8sfh_4n0879ll031cvbsv380000gn/T/ipykernel_47392/3832332138.py:4: RuntimeWarning: Mean of empty slice
  return np.nanmean(x) if AGG_MODE == "mean" else np.nanmedian(x)


In [9]:
print(to_latex_multiindex(
    t_vrmse,
    caption=(
        r"VRMSE $\downarrow$ per PDE and rollout horizon. "
        + HZ_CAPTION_NOTE
        + r" \textbf{Bold}: best; \underline{underline}: 2nd best within each PDE block."
    ),
    label="tab:appendix_vrmse",
    header_lines=[r"PDE & Model & Short & Mid & Long \\"],
))

\begin{table}[t]
\centering
\small
\begin{tabular}{llccc}
\toprule
PDE & Model & Short & Mid & Long \\
\midrule
\multirow{14}{*}{\rotatebox{90}{Turbulence}} & WFM-1 haar & \underline{0.545} & 0.765 & 0.782 \\
 & WFM-1 db2 & 0.553 & 0.752 & 0.856 \\
 & WFM-1 db4 & 0.556 & \underline{0.731} & \textbf{0.776} \\
 & WFM-1 db6 & \textbf{0.544} & \textbf{0.705} & 0.787 \\
 & WFM-3 haar & 0.557 & 0.859 & 0.810 \\
 & WFM-3 db2 & 0.635 & 0.892 & 0.934 \\
 & WFM-3 db4 & 0.578 & 0.763 & 0.817 \\
 & WFM-3 db6 & 0.582 & 0.893 & 0.821 \\
 & FM$_\text{pixel}$ & 0.576 & 0.920 & 0.865 \\
 & FM$_\text{fourier}$ & 0.602 & 0.740 & \underline{0.779} \\
 & WDNO & 1.517 & 1.835 & 1.953 \\
 & WNO & inf & inf & -- \\
 & FNO & 2.952 & 4.978 & 4.349 \\
 & TFNO & 2.300 & 6.534 & 6.387 \\
\midrule
\multirow{14}{*}{\rotatebox{90}{Rayleigh--B\'{e}nard}} & WFM-1 haar & 1.778 & 1.985 & 0.811 \\
 & WFM-1 db2 & 1.326 & 1.893 & 0.797 \\
 & WFM-1 db4 & 1.732 & 2.362 & 0.800 \\
 & WFM-1 db6 & 1.463 & 1.998 & 0.980 \\
 & WFM

## A.2  CRPS — WFM + Generative Models

In [10]:
t_crps = build_main_table(metric="crps", models=WFM_all + gen_models)
t_crps.style.apply(highlight_min_pde, axis=None).format("{:.3f}", na_rep="--")

In [11]:
print(to_latex_multiindex(
    t_crps,
    caption=(
        r"CRPS $\downarrow$ for generative and WFM models. " + HZ_CAPTION_NOTE
    ),
    label="tab:appendix_crps",
    header_lines=[r"PDE & Model & Short & Mid & Long \\"],
))

\begin{table}[t]
\centering
\small
\begin{tabular}{llccc}
\toprule
PDE & Model & Short & Mid & Long \\
\midrule
\multirow{11}{*}{\rotatebox{90}{Turbulence}} & WFM-1 haar & 0.503 & 1.049 & 1.193 \\
 & WFM-1 db2 & 0.449 & 1.144 & 1.427 \\
 & WFM-1 db4 & 0.532 & 1.079 & 1.228 \\
 & WFM-1 db6 & 0.451 & \underline{1.036} & 1.197 \\
 & WFM-3 haar & \textbf{0.415} & 1.073 & \underline{1.173} \\
 & WFM-3 db2 & 0.450 & 1.143 & 1.227 \\
 & WFM-3 db4 & 0.426 & \textbf{0.994} & \textbf{1.124} \\
 & WFM-3 db6 & 0.443 & 1.041 & 1.180 \\
 & FM$_\text{pixel}$ & \underline{0.422} & 1.093 & 1.236 \\
 & FM$_\text{fourier}$ & 0.854 & 1.391 & 1.528 \\
 & WDNO & 2.460 & 3.372 & 4.285 \\
\midrule
\multirow{11}{*}{\rotatebox{90}{Rayleigh--B\'{e}nard}} & WFM-1 haar & 0.005 & 0.036 & 0.078 \\
 & WFM-1 db2 & 0.005 & 0.036 & \underline{0.072} \\
 & WFM-1 db4 & 0.007 & 0.041 & 0.072 \\
 & WFM-1 db6 & 0.005 & 0.040 & 0.091 \\
 & WFM-3 haar & \underline{0.004} & 0.036 & 0.077 \\
 & WFM-3 db2 & \textbf{0.003} & \text

## A.3  Spectral Coherence RMSE — WFM + Generative Models

Values averaged over all rollout horizons within each PDE.

In [12]:
t_spectral = build_spectral_table(models=WFM_all + gen_models)
t_spectral.style.apply(highlight_min_pde, axis=None).format("{:.3f}", na_rep="--")

In [ ]:
_spec_header = [
    r"\multicolumn{2}{l}{} & \multicolumn{3}{c}{Spectral Coherence RMSE} \\",
    r"\cmidrule(lr){3-5}",
    r"PDE & Model & Low & Mid & High \\",
]
print(to_latex_multiindex(
    t_spectral,
    caption=(
        r"Spectral coherence RMSE $\downarrow$ (averaged over rollout horizons) "
        r"for Low / Mid / High frequency bands."
    ),
    label="tab:appendix_spectral",
    header_lines=_spec_header,
))

## A.3b  Spectral Coherence RMSE — by Band × Horizon (detailed)

Optional detailed breakdown: columns are `(band, horizon)` MultiIndex.

In [ ]:
t_spec_hz = build_spectral_horizon_table(models=WFM_all + gen_models)
t_spec_hz.style.apply(highlight_min_pde, axis=None).format("{:.3f}", na_rep="--")

## A.4  Wasserstein-1 Distance — WFM + Generative Models

In [ ]:
t_w1 = build_w1_table(models=WFM_all + gen_models)
t_w1.style.apply(highlight_min_pde, axis=None).format("{:.3f}", na_rep="--")

In [ ]:
print(to_latex_multiindex(
    t_w1,
    caption=r"Wasserstein-1 distance $W_1$ $\downarrow$ for generative and WFM models.",
    label="tab:appendix_w1",
    header_lines=[r"PDE & Model & $W_1$ \\"],
))

---
# B. Ablation Tables

**Rows**: model display labels (WFM-1 variants, WFM-3 variants, generative baselines).
**Columns**: values of the physical parameter being swept.
Three metrics per parameter: **VRMSE** (all models), **CRPS** (generative + WFM), **Spectral RMSE** by band.

| Section | PDE              | Parameter   | New vs original |
|---------|------------------|-------------|-----------------|
| B.1     | Turbulence       | tcool       | + VRMSE (new)   |
| B.2     | Rayleigh–Bénard  | Ra          | + VRMSE (new)   |
| B.3     | Rayleigh–Bénard  | Pr          | + VRMSE (new)   |
| B.4     | Active Matter    | alpha       | + VRMSE (new)   |
| B.5     | Active Matter    | zeta        | + VRMSE (new)   |

## B.1  Turbulent Radiative Layer — $t_\mathrm{cool}$

In [ ]:
_PDE  = "turbulent_radiative_layer_2D"
_PAR  = "tcool"
_FILT = lambda v: v <= 0.030001   # keep only the three standard tcool values
_pde_label = PDE_LABELS[_PDE]

# -- VRMSE --------------------------------------------------------------------
t = build_ablation_table("vrmse", _PDE, _PAR, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"VRMSE $\downarrow$ vs $t_{\mathrm{cool}}$ on Turbulence.",
    label="tab:ablation_trl_vrmse_tcool",
))

In [ ]:
# -- CRPS ---------------------------------------------------------------------
t = build_ablation_table("crps", _PDE, _PAR, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"CRPS $\downarrow$ vs $t_{\mathrm{cool}}$ on Turbulence.",
    label="tab:ablation_trl_crps_tcool",
))

In [ ]:
# -- Spectral RMSE ------------------------------------------------------------
t = build_spectral_ablation_table(_PDE, _PAR, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"Spectral coherence RMSE $\downarrow$ vs $t_{\mathrm{cool}}$ on Turbulence.",
    label="tab:ablation_trl_spectral_tcool",
))

## B.2  Rayleigh–Bénard — Rayleigh Number $Ra$

Prandtl number aggregated out (mean over all Pr values in the sweep).

In [ ]:
_PDE = "rayleigh_benard"
_PAR = "Rayleigh"
_pde_label = PDE_LABELS[_PDE]

# -- VRMSE --------------------------------------------------------------------
t = build_ablation_table("vrmse", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"VRMSE $\downarrow$ vs Rayleigh number $Ra$ on Rayleigh--B\'enard (averaged over $Pr$).",
    label="tab:ablation_rb_vrmse_ra",
))

In [ ]:
# -- CRPS ---------------------------------------------------------------------
t = build_ablation_table("crps", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"CRPS $\downarrow$ vs $Ra$ on Rayleigh--B\'enard (averaged over $Pr$).",
    label="tab:ablation_rb_crps_ra",
))

In [ ]:
# -- Spectral RMSE ------------------------------------------------------------
t = build_spectral_ablation_table(_PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"Spectral coherence RMSE $\downarrow$ vs $Ra$ on Rayleigh--B\'enard (averaged over $Pr$).",
    label="tab:ablation_rb_spectral_ra",
))

## B.3  Rayleigh–Bénard — Prandtl Number $Pr$

Rayleigh number aggregated out (mean over all Ra values in the sweep).

In [ ]:
_PDE = "rayleigh_benard"
_PAR = "Prandtl"
_pde_label = PDE_LABELS[_PDE]

# -- VRMSE --------------------------------------------------------------------
t = build_ablation_table("vrmse", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"VRMSE $\downarrow$ vs $Pr$ on Rayleigh--B\'enard (averaged over $Ra$).",
    label="tab:ablation_rb_vrmse_pr",
))

In [ ]:
# -- CRPS ---------------------------------------------------------------------
t = build_ablation_table("crps", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"CRPS $\downarrow$ vs $Pr$ on Rayleigh--B\'enard (averaged over $Ra$).",
    label="tab:ablation_rb_crps_pr",
))

In [ ]:
# -- Spectral RMSE ------------------------------------------------------------
t = build_spectral_ablation_table(_PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"Spectral coherence RMSE $\downarrow$ vs $Pr$ on Rayleigh--B\'enard (averaged over $Ra$).",
    label="tab:ablation_rb_spectral_pr",
))

## B.4  Active Matter — $\alpha$

Zeta aggregated out. Filtered to the three main sweep values: $\alpha \in \{-3, -4, -5\}$.

In [ ]:
_PDE  = "active_matter"
_PAR  = "alpha"
_FILT = lambda v: v in [-3, -4, -5]
_pde_label = PDE_LABELS[_PDE]

# -- VRMSE --------------------------------------------------------------------
t = build_ablation_table("vrmse", _PDE, _PAR, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"VRMSE $\downarrow$ vs $\alpha$ on Active Matter (averaged over $\zeta$).",
    label="tab:ablation_am_vrmse_alpha",
))

In [ ]:
# -- CRPS ---------------------------------------------------------------------
t = build_ablation_table("crps", _PDE, _PAR, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"CRPS $\downarrow$ vs $\alpha$ on Active Matter (averaged over $\zeta$).",
    label="tab:ablation_am_crps_alpha",
))

In [ ]:
# -- Spectral RMSE ------------------------------------------------------------
t = build_spectral_ablation_table(_PDE, _PAR, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"Spectral coherence RMSE $\downarrow$ vs $\alpha$ on Active Matter (averaged over $\zeta$).",
    label="tab:ablation_am_spectral_alpha",
))

## B.5  Active Matter — $\zeta$

Alpha aggregated out (mean over all alpha values in the sweep).

In [ ]:
_PDE = "active_matter"
_PAR = "zeta"
_pde_label = PDE_LABELS[_PDE]

# -- VRMSE --------------------------------------------------------------------
t = build_ablation_table("vrmse", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"VRMSE $\downarrow$ vs $\zeta$ on Active Matter (averaged over $\alpha$).",
    label="tab:ablation_am_vrmse_zeta",
))

In [ ]:
# -- CRPS ---------------------------------------------------------------------
t = build_ablation_table("crps", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"CRPS $\downarrow$ vs $\zeta$ on Active Matter (averaged over $\alpha$).",
    label="tab:ablation_am_crps_zeta",
))

In [ ]:
# -- Spectral RMSE ------------------------------------------------------------
t = build_spectral_ablation_table(_PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"Spectral coherence RMSE $\downarrow$ vs $\zeta$ on Active Matter (averaged over $\alpha$).",
    label="tab:ablation_am_spectral_zeta",
))